# Haloscan — Model Training (Kaggle CPU)

**Congressional App Challenge** · Trains DualViewNet + HaloscanNet on CPU

Before running:
1. **Settings → Internet** ON (GPU not required)
2. **Optional:** Add-ons → Secrets → `HF_TOKEN` for auto-upload to Hugging Face Hub

Only uses `/kaggle/working/` — does not touch your other Kaggle notebooks/datasets.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path

WORK = Path('/kaggle/working')
SRC = WORK / 'haloscan-src'
REPO = 'https://github.com/arjunkshah12345-hash/haloscan.git'

if not SRC.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO, str(SRC)], check=True)
else:
    subprocess.run(['git', '-C', str(SRC), 'pull', '--ff-only'], check=False)

sys.path.insert(0, str(SRC))
print('Source:', SRC)
print('GPU available:', __import__('torch').cuda.is_available())

In [ ]:
import torch
from haloscan.classifier import train_models, save_models

DEVICE = 'cpu'  # CPU training — no GPU needed
print(f'Training on {DEVICE}')

from haloscan import classifier
orig_build = classifier.build_dataset

def cpu_build(n_per_class=150, size=224, dual=True):
    return orig_build(n_per_class=n_per_class, size=size, dual=dual)

classifier.build_dataset = cpu_build

EPOCHS = 8
BATCH = 32
single, dual = train_models(epochs=EPOCHS, batch_size=BATCH, device=DEVICE)

weights_path = WORK / 'haloscan.pt'
save_models(single, dual, weights_path)
print(f'Saved → {weights_path} ({weights_path.stat().st_size / 1024:.0f} KB)')

In [ ]:
import os, json
from haloscan.evaluate import evaluate_on_synthetic
import haloscan.inference as inf

os.environ['COINCELL_WEIGHTS'] = str(WORK / 'haloscan.pt')
inf._engine = None

metrics = evaluate_on_synthetic(n=40)
metrics_path = WORK / 'metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))
print('✓ Output: haloscan.pt + metrics.json')

In [ ]:
# Done — download haloscan.pt from Kaggle Output tab
# Copy to your repo: weights/haloscan.pt
print(f'Output files ready in {WORK}')
print('Download haloscan.pt → place in weights/haloscan.pt → git push')